# R2 Human-Audit Chance-Corrected Aspect Agreement

Canonical seed-2025 analysis for Table 5 and Supplementary Table S10. It
reports three-level exact agreement, nominal kappa, linear-weighted kappa,
binary-presence agreement, and binary kappa for the two non-expert annotators
and the AI aspect ratings. The post is the bootstrap unit.

Run all cells in Colab. The notebook displays only aggregate agreement tables
and input/QC summaries; it does not display post text.


In [ ]:
# Colab setup and Drive paths
%pip -q install pandas numpy scipy scikit-learn matplotlib openpyxl

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

PROJECTS_ROOT = Path("/content/drive/MyDrive/NLP_Projects")
MENTAL_HEALTH_ROOT = PROJECTS_ROOT / "02_MentalHealth"
R2_ROOT = MENTAL_HEALTH_ROOT / "03_R2"
HUMAN_AUDIT_DIR = MENTAL_HEALTH_ROOT / "02_R1/01_human_validation_300"
required = [
    HUMAN_AUDIT_DIR / "outputs/human_validation_annotator1_300.xlsm",
    HUMAN_AUDIT_DIR / "outputs/human_validation_annotator2_300.xlsm",
    HUMAN_AUDIT_DIR / "generated_subset/human_validation_sample_key_300.csv",
]
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
print("Human-audit inputs: found")


In [ ]:
%%writefile /content/compute_human_aspect_kappas.py
#!/usr/bin/env python3
"""Recompute chance-corrected human-audit aspect agreement.

The bootstrap resampling unit is the sampled post.  Each replicate therefore
keeps all six aspect ratings for a sampled post together and resamples all 300
posts with replacement.
"""

from __future__ import annotations

import hashlib
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd


BASE_DIR = Path(os.environ["MENTAL_HEALTH_HUMAN_AUDIT_DIR"])
ANNOTATOR_1_PATH = BASE_DIR / "outputs" / "human_validation_annotator1_300.xlsm"
ANNOTATOR_2_PATH = BASE_DIR / "outputs" / "human_validation_annotator2_300.xlsm"
SAMPLE_KEY_PATH = BASE_DIR / "generated_subset" / "human_validation_sample_key_300.csv"
OUTPUT_DIR = BASE_DIR / "outputs_notebook"

OVERALL_OUTPUT_PATH = OUTPUT_DIR / "human_validation_aspect_agreement_kappa_overall.csv"
BY_ASPECT_OUTPUT_PATH = OUTPUT_DIR / "human_validation_aspect_agreement_kappa_by_aspect.csv"
MANIFEST_OUTPUT_PATH = OUTPUT_DIR / "human_validation_aspect_agreement_kappa_manifest.json"

ASPECTS = (
    "depression",
    "anxiety",
    "suicidal",
    "stress",
    "bipolar",
    "personality_disorder",
)
ASPECT_DISPLAY = {
    "depression": "Depression",
    "anxiety": "Anxiety",
    "suicidal": "Suicidal",
    "stress": "Stress",
    "bipolar": "Bipolar",
    "personality_disorder": "Personality Disorder",
}
LEVEL_TO_INT = {"none": 0, "weak": 1, "clear": 2}
METRIC_NAMES = (
    "three_level_exact_agreement",
    "three_level_nominal_kappa",
    "three_level_linear_weighted_kappa",
    "binary_exact_agreement",
    "binary_nominal_kappa",
)
COMPARISONS = (
    ("Annotator 1 vs. Annotator 2", "a1", "a2"),
    ("Annotator 1 vs. AI aspects", "a1", "ai"),
    ("Annotator 2 vs. AI aspects", "a2", "ai"),
)
BOOTSTRAP_REPLICATES = 2_000
BOOTSTRAP_SEED = 2025


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def normalize_levels(series: pd.Series, field_name: str) -> pd.Series:
    normalized = series.astype("string").str.strip().str.lower()
    invalid = normalized.isna() | ~normalized.isin(LEVEL_TO_INT)
    if invalid.any():
        values = sorted(normalized.loc[invalid].astype(str).unique().tolist())
        raise AssertionError(f"Invalid values in {field_name}: {values}")
    return normalized


def normalize_boolean(series: pd.Series, field_name: str) -> pd.Series:
    mapping = {
        True: True,
        False: False,
        1: True,
        0: False,
        "true": True,
        "false": False,
        "yes": True,
        "no": False,
    }
    normalized = series.map(lambda value: mapping.get(value, mapping.get(str(value).strip().lower())))
    if normalized.isna().any():
        values = sorted(series.loc[normalized.isna()].astype(str).unique().tolist())
        raise AssertionError(f"Invalid boolean values in {field_name}: {values}")
    return normalized.astype(bool)


def read_inputs() -> tuple[pd.DataFrame, dict[str, object]]:
    a1_raw = pd.read_excel(ANNOTATOR_1_PATH, sheet_name="Annotation")
    a2_raw = pd.read_excel(ANNOTATOR_2_PATH, sheet_name="Annotation")
    key_raw = pd.read_csv(SAMPLE_KEY_PATH)

    for name, frame in (("annotator_1", a1_raw), ("annotator_2", a2_raw), ("sample_key", key_raw)):
        if len(frame) != 300:
            raise AssertionError(f"{name} has {len(frame)} rows, expected 300")
        if frame["sample_id"].isna().any() or frame["sample_id"].nunique() != 300:
            raise AssertionError(f"{name} does not contain 300 unique, nonmissing sample IDs")

    expected_ids = set(a1_raw["sample_id"])
    if set(a2_raw["sample_id"]) != expected_ids or set(key_raw["sample_id"]) != expected_ids:
        raise AssertionError("The three inputs do not contain identical sample-ID sets")

    a1_columns = ["sample_id", "statement"] + [f"human_{aspect}" for aspect in ASPECTS]
    a2_columns = ["sample_id", "statement"] + [f"human_{aspect}" for aspect in ASPECTS]
    key_columns = ["sample_id", "statement"]
    for aspect in ASPECTS:
        key_columns.extend((f"u_{aspect}_strength", f"u_{aspect}_present"))

    a1 = a1_raw[a1_columns].copy().rename(columns={"statement": "statement_a1"})
    a2 = a2_raw[a2_columns].copy().rename(columns={"statement": "statement_a2"})
    key = key_raw[key_columns].copy().rename(columns={"statement": "statement_key"})
    a1 = a1.rename(columns={f"human_{aspect}": f"a1_{aspect}" for aspect in ASPECTS})
    a2 = a2.rename(columns={f"human_{aspect}": f"a2_{aspect}" for aspect in ASPECTS})
    key = key.rename(columns={f"u_{aspect}_strength": f"ai_{aspect}" for aspect in ASPECTS})

    merged = a1.merge(a2, on="sample_id", how="inner", validate="one_to_one")
    merged = merged.merge(key, on="sample_id", how="inner", validate="one_to_one")
    merged = merged.sort_values("sample_id", kind="stable").reset_index(drop=True)
    if len(merged) != 300:
        raise AssertionError(f"Merged audit has {len(merged)} rows, expected 300")
    if not merged["statement_a1"].fillna("").equals(merged["statement_key"].fillna("")):
        raise AssertionError("Annotator 1 workbook and sample-key statements do not match")
    if merged[["statement_a1", "statement_a2", "statement_key"]].isna().any().any():
        raise AssertionError("One or more audit statements are missing")
    a2_statement_mismatch = merged["statement_a2"].ne(merged["statement_key"])

    for source in ("a1", "a2", "ai"):
        for aspect in ASPECTS:
            column = f"{source}_{aspect}"
            merged[column] = normalize_levels(merged[column], column)

    ai_presence_consistent = True
    for aspect in ASPECTS:
        present_column = f"u_{aspect}_present"
        present = normalize_boolean(merged[present_column], present_column)
        expected_present = merged[f"ai_{aspect}"].ne("none")
        if not np.array_equal(present.to_numpy(dtype=bool), expected_present.to_numpy(dtype=bool)):
            ai_presence_consistent = False
            break
    if not ai_presence_consistent:
        raise AssertionError("AI strength and presence fields are inconsistent")

    validation = {
        "annotator_1_rows": int(len(a1_raw)),
        "annotator_2_rows": int(len(a2_raw)),
        "sample_key_rows": int(len(key_raw)),
        "merged_rows": int(len(merged)),
        "unique_sample_ids": int(merged["sample_id"].nunique()),
        "identical_id_sets": True,
        "annotator_1_statements_matching_sample_key": int(merged["statement_a1"].eq(merged["statement_key"]).sum()),
        "annotator_2_statements_matching_sample_key": int((~a2_statement_mismatch).sum()),
        "annotator_2_statement_mismatch_ids": merged.loc[a2_statement_mismatch, "sample_id"].tolist(),
        "all_statements_nonmissing": True,
        "all_strength_values_valid": True,
        "ai_strength_presence_consistent": True,
    }
    return merged, validation


def cohen_kappa(first: np.ndarray, second: np.ndarray, n_levels: int, *, linear: bool) -> float:
    first = np.asarray(first, dtype=np.int8)
    second = np.asarray(second, dtype=np.int8)
    if first.shape != second.shape or first.size == 0:
        raise ValueError("Kappa inputs must be nonempty arrays with identical shapes")

    confusion = np.zeros((n_levels, n_levels), dtype=np.float64)
    np.add.at(confusion, (first, second), 1.0)
    expected = np.outer(confusion.sum(axis=1), confusion.sum(axis=0)) / first.size
    if linear:
        positions = np.arange(n_levels, dtype=np.float64)
        weights = np.abs(positions[:, None] - positions[None, :]) / (n_levels - 1)
    else:
        weights = np.ones((n_levels, n_levels), dtype=np.float64) - np.eye(n_levels)

    observed_disagreement = float(np.sum(weights * confusion))
    expected_disagreement = float(np.sum(weights * expected))
    if np.isclose(expected_disagreement, 0.0):
        return float("nan")
    return 1.0 - observed_disagreement / expected_disagreement


def metrics(first: np.ndarray, second: np.ndarray) -> np.ndarray:
    first = np.asarray(first, dtype=np.int8)
    second = np.asarray(second, dtype=np.int8)
    first_binary = (first > 0).astype(np.int8)
    second_binary = (second > 0).astype(np.int8)
    return np.asarray(
        [
            np.mean(first == second),
            cohen_kappa(first, second, 3, linear=False),
            cohen_kappa(first, second, 3, linear=True),
            np.mean(first_binary == second_binary),
            cohen_kappa(first_binary, second_binary, 2, linear=False),
        ],
        dtype=np.float64,
    )


def metric_summary(point: np.ndarray, bootstrap: np.ndarray) -> dict[str, float | int]:
    result: dict[str, float | int] = {}
    for metric_index, metric_name in enumerate(METRIC_NAMES):
        values = bootstrap[:, metric_index]
        finite = values[np.isfinite(values)]
        if finite.size == 0:
            raise AssertionError(f"No finite bootstrap estimates for {metric_name}")
        result[metric_name] = float(point[metric_index])
        result[f"{metric_name}_ci_low"] = float(np.percentile(finite, 2.5))
        result[f"{metric_name}_ci_high"] = float(np.percentile(finite, 97.5))
        result[f"{metric_name}_bootstrap_valid_n"] = int(finite.size)
    return result


def calculate_agreement(merged: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    encoded: dict[str, np.ndarray] = {}
    for source in ("a1", "a2", "ai"):
        encoded[source] = np.column_stack(
            [merged[f"{source}_{aspect}"].map(LEVEL_TO_INT).to_numpy(dtype=np.int8) for aspect in ASPECTS]
        )

    rng = np.random.default_rng(BOOTSTRAP_SEED)
    bootstrap_indices = rng.integers(0, len(merged), size=(BOOTSTRAP_REPLICATES, len(merged)))
    overall_rows: list[dict[str, object]] = []
    aspect_rows: list[dict[str, object]] = []

    for comparison, first_source, second_source in COMPARISONS:
        first = encoded[first_source]
        second = encoded[second_source]
        point_by_aspect = np.vstack([metrics(first[:, index], second[:, index]) for index in range(len(ASPECTS))])
        point_macro = np.mean(point_by_aspect, axis=0)
        point_micro = metrics(first.reshape(-1), second.reshape(-1))

        bootstrap_by_aspect = np.empty((BOOTSTRAP_REPLICATES, len(ASPECTS), len(METRIC_NAMES)))
        bootstrap_micro = np.empty((BOOTSTRAP_REPLICATES, len(METRIC_NAMES)))
        for replicate, indices in enumerate(bootstrap_indices):
            first_sample = first[indices]
            second_sample = second[indices]
            for aspect_index in range(len(ASPECTS)):
                bootstrap_by_aspect[replicate, aspect_index] = metrics(
                    first_sample[:, aspect_index], second_sample[:, aspect_index]
                )
            bootstrap_micro[replicate] = metrics(first_sample.reshape(-1), second_sample.reshape(-1))
        bootstrap_macro = np.mean(bootstrap_by_aspect, axis=1)

        for aspect_index, aspect in enumerate(ASPECTS):
            row: dict[str, object] = {
                "comparison": comparison,
                "aspect": ASPECT_DISPLAY[aspect],
                "n_items": int(len(merged)),
                "bootstrap_replicates": BOOTSTRAP_REPLICATES,
                "bootstrap_seed": BOOTSTRAP_SEED,
            }
            row.update(metric_summary(point_by_aspect[aspect_index], bootstrap_by_aspect[:, aspect_index]))
            aspect_rows.append(row)

        for aggregation, point, bootstrap in (
            ("macro_across_six_aspects", point_macro, bootstrap_macro),
            ("micro_across_1800_cells", point_micro, bootstrap_micro),
        ):
            row = {
                "comparison": comparison,
                "aggregation": aggregation,
                "n_items": int(len(merged)),
                "n_aspects": int(len(ASPECTS)),
                "n_cells": int(len(merged) * len(ASPECTS)),
                "bootstrap_replicates": BOOTSTRAP_REPLICATES,
                "bootstrap_seed": BOOTSTRAP_SEED,
            }
            row.update(metric_summary(point, bootstrap))
            overall_rows.append(row)

    return pd.DataFrame(overall_rows), pd.DataFrame(aspect_rows)


def assert_acceptance_values(overall: pd.DataFrame) -> dict[str, object]:
    expected_macro = {
        "Annotator 1 vs. Annotator 2": (0.77166667, 0.48746057, 0.55652075, 0.81722222, 0.55451895),
        "Annotator 1 vs. AI aspects": (0.75222222, 0.49689069, 0.55762968, 0.81666667, 0.58846506),
        "Annotator 2 vs. AI aspects": (0.72555556, 0.40927647, 0.49071026, 0.79833333, 0.51235876),
    }
    expected_exact_counts = {
        "Annotator 1 vs. Annotator 2": (1389, 1471),
        "Annotator 1 vs. AI aspects": (1354, 1470),
        "Annotator 2 vs. AI aspects": (1306, 1437),
    }

    macro = overall.loc[overall["aggregation"].eq("macro_across_six_aspects")].set_index("comparison")
    for comparison, expected in expected_macro.items():
        observed = macro.loc[comparison, list(METRIC_NAMES)].to_numpy(dtype=float)
        if not np.allclose(observed, np.asarray(expected), rtol=0.0, atol=5e-8):
            raise AssertionError(f"Acceptance metrics changed for {comparison}: {observed}")
        exact_count = int(round(float(observed[0]) * 1800))
        binary_count = int(round(float(observed[3]) * 1800))
        if (exact_count, binary_count) != expected_exact_counts[comparison]:
            raise AssertionError(f"Acceptance counts changed for {comparison}")

    return {
        "status": "passed",
        "tolerance": 5e-8,
        "expected_macro_metric_order": list(METRIC_NAMES),
        "expected_macro_values": {key: list(value) for key, value in expected_macro.items()},
        "expected_exact_cell_counts": {
            key: {"three_level": value[0], "binary": value[1]}
            for key, value in expected_exact_counts.items()
        },
    }


def main() -> None:
    merged, validation = read_inputs()
    overall, by_aspect = calculate_agreement(merged)
    acceptance = assert_acceptance_values(overall)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    overall.to_csv(OVERALL_OUTPUT_PATH, index=False, float_format="%.6f")
    by_aspect.to_csv(BY_ASPECT_OUTPUT_PATH, index=False, float_format="%.6f")

    manifest = {
        "analysis": "Human agreement audit: chance-corrected aspect agreement",
        "input_files": {
            str(ANNOTATOR_1_PATH.relative_to(BASE_DIR)): sha256(ANNOTATOR_1_PATH),
            str(ANNOTATOR_2_PATH.relative_to(BASE_DIR)): sha256(ANNOTATOR_2_PATH),
            str(SAMPLE_KEY_PATH.relative_to(BASE_DIR)): sha256(SAMPLE_KEY_PATH),
        },
        "validation": validation,
        "comparisons": [comparison for comparison, _, _ in COMPARISONS],
        "aspects": [ASPECT_DISPLAY[aspect] for aspect in ASPECTS],
        "definitions": {
            "three_level_scale": {"none": 0, "weak": 1, "clear": 2},
            "binary_presence": "none=0; weak or clear=1",
            "three_level_exact_agreement": "identical NONE/WEAK/CLEAR ratings",
            "three_level_nominal_kappa": "unweighted Cohen's kappa on three levels",
            "three_level_linear_weighted_kappa": "linear-weighted Cohen's kappa on the ordered three-level scale",
            "binary_nominal_kappa": "unweighted Cohen's kappa after the binary presence mapping",
            "macro": "arithmetic mean of the six aspect-specific estimates",
            "micro": "estimate after pooling all 300 x 6 aspect cells",
        },
        "bootstrap": {
            "unit": "sampled post",
            "scheme": "sample 300 posts with replacement and retain all six aspect cells per selected post",
            "replicates": BOOTSTRAP_REPLICATES,
            "seed": BOOTSTRAP_SEED,
            "interval": "2.5th and 97.5th percentile item-cluster bootstrap interval",
        },
        "acceptance_assertions": acceptance,
        "outputs": {
            str(OVERALL_OUTPUT_PATH.relative_to(BASE_DIR)): int(len(overall)),
            str(BY_ASPECT_OUTPUT_PATH.relative_to(BASE_DIR)): int(len(by_aspect)),
        },
    }
    MANIFEST_OUTPUT_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")

    print(f"Wrote {OVERALL_OUTPUT_PATH}")
    print(f"Wrote {BY_ASPECT_OUTPUT_PATH}")
    print(f"Wrote {MANIFEST_OUTPUT_PATH}")


if __name__ == "__main__":
    main()


In [ ]:
env = os.environ.copy()
env["MENTAL_HEALTH_HUMAN_AUDIT_DIR"] = str(HUMAN_AUDIT_DIR)
run = subprocess.run(
    [sys.executable, "/content/compute_human_aspect_kappas.py"],
    env=env,
    text=True,
    capture_output=True,
)
if run.returncode:
    raise RuntimeError(run.stderr)
print("Chance-corrected human-aspect agreement completed (seed=2025).")


In [ ]:
output_dir = HUMAN_AUDIT_DIR / "outputs_notebook"
overall = pd.read_csv(output_dir / "human_validation_aspect_agreement_kappa_overall.csv")
by_aspect = pd.read_csv(output_dir / "human_validation_aspect_agreement_kappa_by_aspect.csv")
manifest = json.loads((output_dir / "human_validation_aspect_agreement_kappa_manifest.json").read_text())

display(overall.round(4))
display(by_aspect.round(4))
display(pd.DataFrame([{
    "n_posts": manifest["validation"]["merged_rows"],
    "n_aspects": len(manifest["aspects"]),
    "bootstrap_replicates": manifest["bootstrap"]["replicates"],
    "bootstrap_seed": manifest["bootstrap"]["seed"],
    "acceptance_status": manifest["acceptance_assertions"]["status"],
}]))
